In [1]:
from __future__ import annotations

from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path
import gc
import shutil
from typing import Any

import pythoncom
import win32com.client as win32

SAMPLES_DIR = Path("samples")
OUTPUT_DIR = Path("outputs")
MASTER_FILE = SAMPLES_DIR / "Expense Report Relating to deals 01-01-26 to 6-30-26_V8 - COMBINED - Try SAP.xlsx"
HR_FILE = SAMPLES_DIR / "Headcount CC 0426.xlsx"
SOURCE_SHEET_NAME = "To be Copy Pasted from HR"
MASTER_SHEET_PREFIX = "HR HC Combined"
NETWORK_ID_HEADER = "Network ID"
EFFECTIVE_DATE_HEADER = "Effective Date"
DUPLICATE_HEADERS = {"DUPLICATE", "DUPLICATES", "DUPLICATE?"}
XL_UP = -4162
XL_TO_LEFT = -4159
XL_PASTE_FORMATS = -4122
XL_CALCULATION_MANUAL = -4135
XL_CALCULATION_AUTOMATIC = -4105


def normalize_header(value: Any) -> str:
    return " ".join(str(value or "").strip().split()).casefold()


def normalize_network_id(value: Any) -> str:
    if value is None or isinstance(value, bool):
        return ""
    text = str(value).strip()
    if text.endswith(".0") and text[:-2].isdigit():
        text = text[:-2]
    return text.casefold()


def coerce_excel_date(value: Any) -> date | None:
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        try:
            return (datetime(1899, 12, 30) + timedelta(days=float(value))).date()
        except (ValueError, OverflowError):
            return None
    text = str(value or "").strip()
    for pattern in ("%m/%d/%Y", "%Y-%m-%d", "%m-%d-%Y"):
        try:
            return datetime.strptime(text, pattern).date()
        except ValueError:
            continue
    return None


def rows_from_range(values: Any) -> list[tuple[Any, ...]]:
    if values in (None, ""):
        return []
    if not isinstance(values, tuple):
        return [(values,)]
    if values and not isinstance(values[0], tuple):
        return [tuple(values)]
    return [tuple(row) for row in values]


def find_header_row(worksheet: Any, required_header: str, maximum_rows: int = 20) -> int:
    last_column = worksheet.UsedRange.Column + worksheet.UsedRange.Columns.Count - 1
    for row_number in range(1, min(maximum_rows, worksheet.UsedRange.Row + worksheet.UsedRange.Rows.Count - 1) + 1):
        values = rows_from_range(worksheet.Range(worksheet.Cells(row_number, 1), worksheet.Cells(row_number, last_column)).Value2)[0]
        if normalize_header(required_header) in {normalize_header(value) for value in values}:
            return row_number
    raise ValueError(f"Could not find header {required_header!r} in the first {maximum_rows} rows of {worksheet.Name!r}.")


def read_headers(worksheet: Any, header_row: int) -> dict[str, int]:
    last_column = worksheet.Cells(header_row, worksheet.Columns.Count).End(XL_TO_LEFT).Column
    values = rows_from_range(worksheet.Range(worksheet.Cells(header_row, 1), worksheet.Cells(header_row, last_column)).Value2)[0]
    headers: dict[str, int] = {}
    duplicates: list[str] = []
    for column, value in enumerate(values, start=1):
        header = normalize_header(value)
        if not header:
            continue
        if header in headers:
            duplicates.append(str(value))
        headers[header] = column
    if duplicates:
        raise ValueError(f"Duplicate nonblank headers in {worksheet.Name!r}: {duplicates}")
    return headers


def resolve_master_sheet(workbook: Any) -> Any:
    matches = [workbook.Worksheets(index) for index in range(1, workbook.Worksheets.Count + 1) if workbook.Worksheets(index).Name.startswith(MASTER_SHEET_PREFIX)]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one worksheet beginning {MASTER_SHEET_PREFIX!r}; found {[sheet.Name for sheet in matches]}.")
    return matches[0]


def last_data_row(worksheet: Any, key_column: int, header_row: int) -> int:
    row = worksheet.Cells(worksheet.Rows.Count, key_column).End(XL_UP).Row
    return max(header_row, row)


def read_data_rows(worksheet: Any, header_row: int, last_row: int, column_count: int, key_column: int, effective_date_column: int, origin: str) -> list[dict[str, Any]]:
    if last_row <= header_row:
        return []
    raw_rows = rows_from_range(worksheet.Range(worksheet.Cells(header_row + 1, 1), worksheet.Cells(last_row, column_count)).Value2)
    rows = []
    for row_number, values in enumerate(raw_rows, start=header_row + 1):
        if all(value in (None, "") for value in values):
            continue
        network_id = normalize_network_id(values[key_column - 1])
        effective_date = coerce_excel_date(values[effective_date_column - 1])
        if network_id and effective_date is None:
            raise ValueError(f"{origin} row {row_number} has Network ID {values[key_column - 1]!r} but no valid {EFFECTIVE_DATE_HEADER}.")
        rows.append({"origin": origin, "row_number": row_number, "values": values, "network_id": network_id, "effective_date": effective_date})
    return rows


def retain_latest_rows(candidates: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    by_network_id: dict[str, list[dict[str, Any]]] = defaultdict(list)
    blank_id_rows = []
    for row in candidates:
        (by_network_id[row["network_id"]] if row["network_id"] else blank_id_rows).append(row)

    retained = list(blank_id_rows)
    removed = []
    ties = []
    for network_id, rows in by_network_id.items():
        latest_date = max(row["effective_date"] for row in rows)
        latest_rows = [row for row in rows if row["effective_date"] == latest_date]
        if len(latest_rows) != 1:
            ties.append(f"{network_id}: " + ", ".join(f"{row['origin']} row {row['row_number']}" for row in latest_rows))
            continue
        retained.append(latest_rows[0])
        removed.extend(row for row in rows if row is not latest_rows[0])
    if ties:
        raise ValueError("Cannot choose a latest record for Network ID(s) with equal maximum Effective Date: " + "; ".join(ties[:20]))
    retained.sort(key=lambda row: (row["origin"] != "master", row["row_number"]))
    return retained, removed


def next_output_path() -> Path:
    OUTPUT_DIR.mkdir(exist_ok=True)
    base = OUTPUT_DIR / f"{MASTER_FILE.stem}_HR_REFRESHED{MASTER_FILE.suffix}"
    candidate = base
    number = 1
    while candidate.exists():
        candidate = base.with_name(f"{base.stem}_{number}{base.suffix}")
        number += 1
    return candidate


def preflight() -> dict[str, Any]:
    if not MASTER_FILE.is_file() or not HR_FILE.is_file():
        raise FileNotFoundError(f"Missing master or HR source: {MASTER_FILE}, {HR_FILE}")
    excel = master_book = source_book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        master_book = excel.Workbooks.Open(str(MASTER_FILE.resolve()), ReadOnly=True)
        source_book = excel.Workbooks.Open(str(HR_FILE.resolve()), ReadOnly=True)
        master_sheet = resolve_master_sheet(master_book)
        source_sheet = source_book.Worksheets(SOURCE_SHEET_NAME)
        master_header_row = find_header_row(master_sheet, NETWORK_ID_HEADER)
        source_header_row = find_header_row(source_sheet, NETWORK_ID_HEADER)
        master_headers = read_headers(master_sheet, master_header_row)
        source_headers = read_headers(source_sheet, source_header_row)
        key = normalize_header(NETWORK_ID_HEADER)
        effective_date_key = normalize_header(EFFECTIVE_DATE_HEADER)
        if key not in master_headers or key not in source_headers:
            raise ValueError("Network ID must occur exactly once in both source and master.")
        if effective_date_key not in master_headers or effective_date_key not in source_headers:
            raise ValueError(f"{EFFECTIVE_DATE_HEADER} must occur exactly once in both source and master.")
        source_only = sorted(header for header in source_headers if header not in master_headers)
        if source_only:
            raise ValueError(f"Source columns missing from {master_sheet.Name!r}: {source_only}")
        duplicate_column = next((column for header, column in master_headers.items() if header in DUPLICATE_HEADERS), None)
        result = {"master_sheet_name": master_sheet.Name, "source_header_row": source_header_row, "master_header_row": master_header_row, "source_headers": source_headers, "master_headers": master_headers, "source_key_column": source_headers[key], "master_key_column": master_headers[key], "source_effective_date_column": source_headers[effective_date_key], "master_effective_date_column": master_headers[effective_date_key], "duplicate_column": duplicate_column}
        print(f"Preflight passed: source={source_sheet.Name!r} row {source_header_row}; master={master_sheet.Name!r} row {master_header_row}.")
        print(f"Mapped HR columns: {len(source_headers)}. Optional duplicate control: {'present' if duplicate_column else 'not present'}.")
        return result
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if master_book is not None:
            master_book.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


def analyse_refresh(configuration: dict[str, Any]) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    excel = master_book = source_book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        master_book = excel.Workbooks.Open(str(MASTER_FILE.resolve()), ReadOnly=True)
        source_book = excel.Workbooks.Open(str(HR_FILE.resolve()), ReadOnly=True)
        master_sheet = resolve_master_sheet(master_book)
        source_sheet = source_book.Worksheets(SOURCE_SHEET_NAME)
        master_rows = read_data_rows(master_sheet, configuration["master_header_row"], last_data_row(master_sheet, configuration["master_key_column"], configuration["master_header_row"]), max(configuration["master_headers"].values()), configuration["master_key_column"], configuration["master_effective_date_column"], "master")
        source_rows = read_data_rows(source_sheet, configuration["source_header_row"], last_data_row(source_sheet, configuration["source_key_column"], configuration["source_header_row"]), max(configuration["source_headers"].values()), configuration["source_key_column"], configuration["source_effective_date_column"], "source")
        retained, removed = retain_latest_rows(master_rows + source_rows)
        print(f"Existing rows: {len(master_rows):,}; source rows: {len(source_rows):,}; retained: {len(retained):,}; superseded: {len(removed):,}.")
        print(f"Blank Network ID rows retained: {sum(not row['network_id'] for row in retained):,}.")
        return retained, removed
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if master_book is not None:
            master_book.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


def refresh_workbook(configuration: dict[str, Any], retained_rows: list[dict[str, Any]]) -> Path:
    output_path = next_output_path()
    shutil.copy2(MASTER_FILE, output_path)
    excel = master_book = None
    original_autofill = None
    succeeded = False
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.ScreenUpdating = False
        excel.EnableEvents = False
        excel.Calculation = XL_CALCULATION_MANUAL
        original_autofill = excel.AutoCorrect.AutoFillFormulasInLists
        excel.AutoCorrect.AutoFillFormulasInLists = False
        master_book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=False)
        worksheet = resolve_master_sheet(master_book)
        headers = configuration["master_headers"]
        header_row = configuration["master_header_row"]
        key_column = configuration["master_key_column"]
        old_last_row = last_data_row(worksheet, key_column, header_row)
        final_row = header_row + len(retained_rows)
        final_column = max(headers.values())
        if old_last_row > header_row:
            worksheet.Range(worksheet.Cells(old_last_row, 1), worksheet.Cells(old_last_row, final_column)).Copy()
            worksheet.Range(worksheet.Cells(header_row + 1, 1), worksheet.Cells(final_row, final_column)).PasteSpecial(Paste=XL_PASTE_FORMATS)
            worksheet.Application.CutCopyMode = False
            worksheet.Range(worksheet.Cells(header_row + 1, 1), worksheet.Cells(max(old_last_row, final_row), final_column)).ClearContents()
        source_headers = configuration["source_headers"]
        for source_header, source_column in source_headers.items():
            master_column = headers[source_header]
            values = []
            for row in retained_rows:
                if row["origin"] == "source":
                    values.append(row["values"][source_column - 1])
                else:
                    values.append(row["values"][master_column - 1])
            worksheet.Range(worksheet.Cells(header_row + 1, master_column), worksheet.Cells(final_row, master_column)).Value2 = tuple((value,) for value in values)
        duplicate_column = configuration["duplicate_column"]
        if duplicate_column:
            template = worksheet.Cells(header_row + 1, duplicate_column).FormulaR1C1
            if isinstance(template, str) and template.startswith("="):
                worksheet.Range(worksheet.Cells(header_row + 1, duplicate_column), worksheet.Cells(final_row, duplicate_column)).FormulaR1C1 = template
                print("Optional DUPLICATE control formula refreshed.")
            else:
                print("Optional DUPLICATE column has no formula template; preserved without recalculation.")
        for index in range(1, worksheet.ListObjects.Count + 1):
            table = worksheet.ListObjects(index)
            if table.Range.Row == header_row and table.Range.Column <= key_column <= table.Range.Column + table.Range.Columns.Count - 1:
                table.Resize(worksheet.Range(worksheet.Cells(header_row, table.Range.Column), worksheet.Cells(final_row, table.Range.Column + table.Range.Columns.Count - 1)))
                break
        excel.Calculation = XL_CALCULATION_AUTOMATIC
        excel.CalculateFull()
        master_book.Save()
        succeeded = True
        return output_path
    finally:
        if master_book is not None:
            master_book.Close(SaveChanges=succeeded)
        if excel is not None:
            if original_autofill is not None:
                excel.AutoCorrect.AutoFillFormulasInLists = original_autofill
            excel.Quit()
        if not succeeded and output_path.exists():
            output_path.unlink()
        gc.collect()
        pythoncom.CoUninitialize()


def validate_output(output_path: Path, expected_rows: list[dict[str, Any]], configuration: dict[str, Any]) -> None:
    excel = book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=True)
        worksheet = resolve_master_sheet(book)
        actual = read_data_rows(worksheet, configuration["master_header_row"], last_data_row(worksheet, configuration["master_key_column"], configuration["master_header_row"]), max(configuration["master_headers"].values()), configuration["master_key_column"], configuration["master_effective_date_column"], "output")
        expected_keys = {(row["network_id"], row["effective_date"]) for row in expected_rows if row["network_id"]}
        actual_keys = {(row["network_id"], row["effective_date"]) for row in actual if row["network_id"]}
        actual_ids = [row["network_id"] for row in actual if row["network_id"]]
        if len(actual_ids) != len(set(actual_ids)) or expected_keys != actual_keys:
            raise AssertionError("Saved workbook does not match the approved unique Network ID + latest Effective Date population.")
        if sum(not row["network_id"] for row in actual) != sum(not row["network_id"] for row in expected_rows):
            raise AssertionError("Saved workbook did not preserve the expected blank Network ID rows.")
        print(f"Post-save validation passed: {len(actual):,} rows; {len(actual_ids):,} unique nonblank Network IDs.")
    finally:
        if book is not None:
            book.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


configuration = preflight()
retained_rows, removed_rows = analyse_refresh(configuration)
print("Review the displayed counts before running the next cell. No workbook has been changed yet.")

Preflight passed: source='To be Copy Pasted from HR' row 4; master='HR HC Combined Jun' row 1.
Mapped HR columns: 24. Optional duplicate control: not present.
Existing rows: 1,624; source rows: 1,400; retained: 1,622; superseded: 1,402.
Blank Network ID rows retained: 0.
Review the displayed counts before running the next cell. No workbook has been changed yet.


In [ ]:
def set_excel_calculation(excel: Any, calculation_mode: int) -> None:
    try:
        excel.Calculation = calculation_mode
    except Exception as error:
        print(f"Excel calculation mode was not changed: {error}")


def refresh_workbook(configuration: dict[str, Any], retained_rows: list[dict[str, Any]]) -> Path:
    output_path = next_output_path()
    shutil.copy2(MASTER_FILE, output_path)
    excel = master_book = None
    original_autofill = None
    succeeded = False
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.ScreenUpdating = False
        excel.EnableEvents = False
        set_excel_calculation(excel, XL_CALCULATION_MANUAL)
        try:
            original_autofill = excel.AutoCorrect.AutoFillFormulasInLists
            excel.AutoCorrect.AutoFillFormulasInLists = False
        except Exception as error:
            print(f"Excel table autofill was not changed: {error}")
        master_book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=False)
        worksheet = resolve_master_sheet(master_book)
        headers = configuration["master_headers"]
        header_row = configuration["master_header_row"]
        key_column = configuration["master_key_column"]
        old_last_row = last_data_row(worksheet, key_column, header_row)
        final_row = header_row + len(retained_rows)
        final_column = max(headers.values())
        duplicate_column = next((column for header, column in headers.items() if header.rstrip("?") in {"duplicate", "duplicates"}), None)
        duplicate_formula = worksheet.Cells(header_row + 1, duplicate_column).FormulaR1C1 if duplicate_column else None
        if old_last_row > header_row:
            worksheet.Range(worksheet.Cells(old_last_row, 1), worksheet.Cells(old_last_row, final_column)).Copy()
            worksheet.Range(worksheet.Cells(header_row + 1, 1), worksheet.Cells(final_row, final_column)).PasteSpecial(Paste=XL_PASTE_FORMATS)
            worksheet.Application.CutCopyMode = False
            worksheet.Range(worksheet.Cells(header_row + 1, 1), worksheet.Cells(max(old_last_row, final_row), final_column)).ClearContents()
        source_headers = configuration["source_headers"]
        for source_header, source_column in source_headers.items():
            master_column = headers[source_header]
            values = [row["values"][source_column - 1] if row["origin"] == "source" else row["values"][master_column - 1] for row in retained_rows]
            worksheet.Range(worksheet.Cells(header_row + 1, master_column), worksheet.Cells(final_row, master_column)).Value2 = tuple((value,) for value in values)
        if duplicate_column and isinstance(duplicate_formula, str) and duplicate_formula.startswith("="):
            worksheet.Range(worksheet.Cells(header_row + 1, duplicate_column), worksheet.Cells(final_row, duplicate_column)).FormulaR1C1 = duplicate_formula
            print("Optional DUPLICATE control formula refreshed.")
        elif duplicate_column:
            print("Optional DUPLICATE column has no formula template; it remains blank for refreshed rows.")
        for index in range(1, worksheet.ListObjects.Count + 1):
            table = worksheet.ListObjects(index)
            if table.Range.Row == header_row and table.Range.Column <= key_column <= table.Range.Column + table.Range.Columns.Count - 1:
                table.Resize(worksheet.Range(worksheet.Cells(header_row, table.Range.Column), worksheet.Cells(final_row, table.Range.Column + table.Range.Columns.Count - 1)))
                break
        set_excel_calculation(excel, XL_CALCULATION_AUTOMATIC)
        excel.CalculateFull()
        master_book.Save()
        succeeded = True
        return output_path
    finally:
        if master_book is not None:
            master_book.Close(SaveChanges=succeeded)
        if excel is not None:
            if original_autofill is not None:
                excel.AutoCorrect.AutoFillFormulasInLists = original_autofill
            excel.Quit()
        if not succeeded and output_path.exists():
            output_path.unlink()
        gc.collect()
        pythoncom.CoUninitialize()

In [ ]:
def validate_output(output_path: Path, expected_rows: list[dict[str, Any]], configuration: dict[str, Any]) -> None:
    excel = book = None
    pythoncom.CoInitialize()
    gc.collect()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=True)
        worksheet = resolve_master_sheet(book)
        actual = read_data_rows(
            worksheet,
            configuration["master_header_row"],
            last_data_row(worksheet, configuration["master_key_column"], configuration["master_header_row"]),
            max(configuration["master_headers"].values()),
            configuration["master_key_column"],
            "output",
        )
        expected_keys = {(row["network_id"], row["effective_date"]) for row in expected_rows if row["network_id"]}
        actual_keys = {(row["network_id"], row["effective_date"]) for row in actual if row["network_id"]}
        actual_ids = [row["network_id"] for row in actual if row["network_id"]]
        if len(actual_ids) != len(set(actual_ids)) or expected_keys != actual_keys:
            raise AssertionError("Saved workbook does not match the approved unique Network ID + latest Effective Date population.")
        if sum(not row["network_id"] for row in actual) != sum(not row["network_id"] for row in expected_rows):
            raise AssertionError("Saved workbook did not preserve the expected blank Network ID rows.")
        print(f"Post-save validation passed: {len(actual):,} rows; {len(actual_ids):,} unique nonblank Network IDs.")
    finally:
        if book is not None:
            try:
                book.Close(SaveChanges=False)
            except Exception:
                pass
        if excel is not None:
            try:
                excel.Quit()
            except Exception:
                pass
        gc.collect()
        pythoncom.CoUninitialize()


configuration["duplicate_column"] = next(
    (column for header, column in configuration["master_headers"].items() if header.rstrip("?") in {"duplicate", "duplicates"}),
    None,
)
print(f"Optional duplicate control: {'present' if configuration['duplicate_column'] else 'not present'}.")

In [ ]:
# Execute only after reviewing the preflight and analysis counts above.
output_path = refresh_workbook(configuration, retained_rows)
validate_output(output_path, retained_rows, configuration)
print(f"Saved validated HR refresh: {output_path}")

In [ ]:
# Uber Report has two protected Cost Center helper columns; raw fields must remain unique.
_original_read_headers = read_headers


def read_headers(worksheet: Any, header_row: int) -> dict[str, int]:
    last_column = worksheet.Cells(header_row, worksheet.Columns.Count).End(XL_TO_LEFT).Column
    values = rows_from_range(worksheet.Range(worksheet.Cells(header_row, 1), worksheet.Cells(header_row, last_column)).Value2)[0]
    headers: dict[str, int] = {}
    duplicates: list[str] = []
    for column, value in enumerate(values, start=1):
        header = normalize_header(value)
        if not header:
            continue
        if header in headers:
            if worksheet.Name == "Uber Report" and header == "cost center":
                continue
            duplicates.append(str(value))
            continue
        headers[header] = column
    if duplicates:
        raise ValueError(f"Duplicate nonblank headers in {worksheet.Name!r}: {duplicates}")
    return headers

In [ ]:
# Support the two protected Cost Center helper columns in Uber Report.
def uber_headers(worksheet: Any, header_row: int) -> dict[str, int]:
    last_column = worksheet.Cells(header_row, worksheet.Columns.Count).End(XL_TO_LEFT).Column
    values = rows_from_range(worksheet.Range(worksheet.Cells(header_row, 1), worksheet.Cells(header_row, last_column)).Value2)[0]
    headers: dict[str, int] = {}
    duplicates: list[str] = []
    for column, value in enumerate(values, start=1):
        header = normalize_header(value)
        if not header:
            continue
        if header in headers:
            if header not in PROTECTED_HELPERS:
                duplicates.append(str(value))
            continue
        headers[header] = column
    if duplicates:
        raise ValueError(f"Duplicate raw-data headers in {worksheet.Name!r}: {duplicates}")
    return headers

In [ ]:
# UBER STEPS 2 AND 3: pre-load validation and raw-data reload.
# This section uses Excel COM exclusively for workbook access and mutations.
from decimal import Decimal, InvalidOperation

UBER_SOURCE_FILE = SAMPLES_DIR / "U4B 1H 2026.xlsx"
UBER_SOURCE_SHEET = "U4B"
UBER_REPORT_SHEET = "Uber Report"
IN_SCOPE_SHEET = "In Scope CCs"
UBER_HEADER_ROW = 6
UBER_FIRST_DATA_ROW = UBER_HEADER_ROW + 1
PROTECTED_HELPERS = {
    "full name",
    "cost center",
    "final cost center",
    "lob",
    "in scope?",
    "in scope",
    "company name",
}
REQUIRED_UBER_SOURCE_HEADERS = {"request date (utc)", "first name", "last name", "employee id", "transaction amount usd"}


def decimal_amount(value: Any) -> Decimal:
    if value in (None, ""):
        return Decimal("0")
    try:
        return Decimal(str(value).replace(",", "").strip())
    except InvalidOperation as error:
        raise ValueError(f"Transaction Amount USD is not numeric: {value!r}") from error


def uber_headers(worksheet: Any, header_row: int) -> dict[str, int]:
    return read_headers(worksheet, header_row)


def require_column(headers: dict[str, int], header: str, sheet_name: str) -> int:
    column = headers.get(normalize_header(header))
    if column is None:
        raise ValueError(f"Required column {header!r} is missing from {sheet_name!r}.")
    return column


def find_uber_lob_mapping(worksheet: Any) -> tuple[int, int, int, int]:
    """Find the second Cost Center/LOB table labelled Uber LOB Mapping."""
    used = worksheet.UsedRange
    final_row = used.Row + used.Rows.Count - 1
    marker_row = None
    for row in range(1, final_row + 1):
        if normalize_header(worksheet.Cells(row, 1).Value2) == "uber lob mapping":
            marker_row = row
            break
    if marker_row is None:
        raise ValueError("Could not find the 'Uber LOB Mapping' section in In Scope CCs.")
    header_row = marker_row + 1
    headers = uber_headers(worksheet, header_row)
    return header_row, require_column(headers, "Cost Center", worksheet.Name), require_column(headers, "LOB", worksheet.Name), final_row


def last_populated_row(worksheet: Any, column: int, header_row: int) -> int:
    return max(header_row, worksheet.Cells(worksheet.Rows.Count, column).End(XL_UP).Row)


def range_rows(worksheet: Any, first_row: int, final_row: int, final_column: int) -> list[tuple[Any, ...]]:
    if final_row < first_row:
        return []
    return rows_from_range(worksheet.Range(worksheet.Cells(first_row, 1), worksheet.Cells(final_row, final_column)).Value2)


def build_hr_lookups(workbook: Any) -> tuple[dict[str, str], dict[str, str], str, int]:
    worksheet = resolve_master_sheet(workbook)
    headers = uber_headers(worksheet, 1)
    id_column = require_column(headers, "Network ID", worksheet.Name)
    name_column = require_column(headers, "Name", worksheet.Name)
    cost_center_column = require_column(headers, "Cost Center Name", worksheet.Name)
    final_row = last_populated_row(worksheet, id_column, 1)
    by_id: dict[str, str] = {}
    name_cost_centers: dict[str, set[str]] = defaultdict(set)
    for row in range_rows(worksheet, 2, final_row, max(headers.values())):
        employee_id = normalize_network_id(row[id_column - 1])
        employee_name = normalize_header(row[name_column - 1])
        cost_center = str(row[cost_center_column - 1] or "").strip()
        if employee_id and cost_center:
            by_id[employee_id] = cost_center
        if employee_name and cost_center:
            name_cost_centers[employee_name].add(cost_center)
    by_name = {name: next(iter(cost_centers)) for name, cost_centers in name_cost_centers.items() if len(cost_centers) == 1}
    return by_id, by_name, worksheet.Name, final_row


def build_lob_lookup(workbook: Any) -> tuple[dict[str, str], int, int, int]:
    worksheet = workbook.Worksheets(IN_SCOPE_SHEET)
    header_row, cost_center_column, lob_column, sheet_final_row = find_uber_lob_mapping(worksheet)
    mapping: dict[str, str] = {}
    row = header_row + 1
    while row <= sheet_final_row:
        cost_center = str(worksheet.Cells(row, cost_center_column).Value2 or "").strip()
        lob = str(worksheet.Cells(row, lob_column).Value2 or "").strip()
        if not cost_center and not lob:
            break
        key = normalize_header(cost_center)
        if key in mapping and mapping[key] != lob:
            raise ValueError(f"Conflicting Uber LOB mapping for {cost_center!r}.")
        if key and lob:
            mapping[key] = lob
        row += 1
    if not mapping:
        raise ValueError("Uber LOB Mapping contains no usable cost center rows.")
    return mapping, header_row, row - 1, cost_center_column


def current_uber_ytd(worksheet: Any, headers: dict[str, int]) -> tuple[Decimal, int, int]:
    """Read the prior total from Uber Report data, avoiding fixed total cells."""
    amount_column = require_column(headers, "Transaction Amount USD", worksheet.Name)
    in_scope_column = require_column(headers, "In scope?", worksheet.Name)
    last_row = last_populated_row(worksheet, amount_column, UBER_HEADER_ROW)
    total = Decimal("0")
    rows = 0
    for row in range_rows(worksheet, UBER_FIRST_DATA_ROW, last_row, max(headers.values())):
        rows += 1
        if normalize_header(row[in_scope_column - 1]) == "yes":
            total += decimal_amount(row[amount_column - 1])
    return total, rows, last_row


def analyse_uber_source() -> dict[str, Any]:
    if not MASTER_FILE.is_file() or not UBER_SOURCE_FILE.is_file():
        raise FileNotFoundError("The master workbook and U4B source workbook are required.")
    excel = master_book = source_book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        master_book = excel.Workbooks.Open(str(MASTER_FILE.resolve()), ReadOnly=True)
        source_book = excel.Workbooks.Open(str(UBER_SOURCE_FILE.resolve()), ReadOnly=True)
        uber_sheet = master_book.Worksheets(UBER_REPORT_SHEET)
        source_sheet = source_book.Worksheets(UBER_SOURCE_SHEET)
        target_headers = uber_headers(uber_sheet, UBER_HEADER_ROW)
        source_headers = uber_headers(source_sheet, 1)
        missing = sorted(REQUIRED_UBER_SOURCE_HEADERS - set(source_headers))
        if missing:
            raise ValueError(f"Uber source is missing required columns: {missing}")
        source_only = sorted(header for header in source_headers if header not in target_headers)
        mapped = {header: target_headers[header] for header in source_headers if header in target_headers and header not in PROTECTED_HELPERS}
        if source_only:
            print(f"Source-only columns ignored because they do not exist in Uber Report: {source_only}")
        protected_present = {header: target_headers[header] for header in PROTECTED_HELPERS if header in target_headers}
        missing_protected = PROTECTED_HELPERS - set(protected_present)
        if missing_protected:
            print(f"Optional protected helpers not present: {sorted(missing_protected)}")
        prior_total, prior_rows, prior_last_row = current_uber_ytd(uber_sheet, target_headers)
        hr_by_id, hr_by_name, hr_sheet_name, hr_last_row = build_hr_lookups(master_book)
        lob_by_cost_center, lob_header_row, lob_last_row, lob_cost_center_column = build_lob_lookup(master_book)
        source_amount_column = require_column(source_headers, "Transaction Amount USD", source_sheet.Name)
        source_id_column = require_column(source_headers, "Employee ID", source_sheet.Name)
        source_first_name_column = require_column(source_headers, "First Name", source_sheet.Name)
        source_last_name_column = require_column(source_headers, "Last Name", source_sheet.Name)
        source_last_row = last_populated_row(source_sheet, source_amount_column, 1)
        source_rows = range_rows(source_sheet, 2, source_last_row, max(source_headers.values()))
        candidate_total = Decimal("0")
        id_matches = name_matches = unmatched = 0
        for row in source_rows:
            employee_id = normalize_network_id(row[source_id_column - 1])
            full_name = normalize_header(f"{row[source_first_name_column - 1] or ''} {row[source_last_name_column - 1] or ''}")
            cost_center = hr_by_id.get(employee_id, "")
            if cost_center:
                id_matches += 1
            else:
                cost_center = hr_by_name.get(full_name, "")
                if cost_center:
                    name_matches += 1
                else:
                    unmatched += 1
            if normalize_header(cost_center) in lob_by_cost_center:
                candidate_total += decimal_amount(row[source_amount_column - 1])
        unmatched_percent = (Decimal(unmatched) * Decimal("100") / Decimal(len(source_rows))) if source_rows else Decimal("0")
        result = {
            "target_headers": target_headers,
            "source_headers": source_headers,
            "mapped_columns": mapped,
            "protected_columns": protected_present,
            "prior_total": prior_total,
            "prior_rows": prior_rows,
            "prior_last_row": prior_last_row,
            "candidate_total": candidate_total,
            "source_rows": source_rows,
            "source_last_row": source_last_row,
            "id_matches": id_matches,
            "name_matches": name_matches,
            "unmatched": unmatched,
            "unmatched_percent": unmatched_percent,
            "hr_sheet_name": hr_sheet_name,
            "hr_last_row": hr_last_row,
            "lob_header_row": lob_header_row,
            "lob_last_row": lob_last_row,
            "lob_cost_center_column": lob_cost_center_column,
        }
        print(f"Uber Report prior in-scope YTD: ${prior_total:,.2f} across {prior_rows:,} rows.")
        print(f"U4B candidate in-scope YTD: ${candidate_total:,.2f} across {len(source_rows):,} imported rows.")
        print(f"Matches: employee ID={id_matches:,}; full-name fallback={name_matches:,}; unmatched={unmatched:,} ({unmatched_percent:.2f}%).")
        if candidate_total < prior_total:
            delta = candidate_total - prior_total
            raise RuntimeError(
                "UBER REFRESH FAILED: Current in-scope YTD is lower than the prior Uber Report YTD. "
                f"Prior=${prior_total:,.2f}; current=${candidate_total:,.2f}; change=${delta:,.2f}. "
                "No output workbook was created."
            )
        print("Pre-load YTD validation passed. The reload cell may now be run.")
        return result
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if master_book is not None:
            master_book.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


uber_configuration = analyse_uber_source()

In [ ]:
def uber_headers(worksheet: Any, header_row: int) -> dict[str, int]:
    """Read Uber headers, allowing the two protected Cost Center helpers only."""
    last_column = worksheet.Cells(header_row, worksheet.Columns.Count).End(XL_TO_LEFT).Column
    values = rows_from_range(worksheet.Range(worksheet.Cells(header_row, 1), worksheet.Cells(header_row, last_column)).Value2)[0]
    headers: dict[str, int] = {}
    duplicates: list[str] = []
    for column, value in enumerate(values, start=1):
        header = normalize_header(value)
        if not header:
            continue
        if header in headers:
            if header not in PROTECTED_HELPERS:
                duplicates.append(str(value))
            continue
        headers[header] = column
    if duplicates:
        raise ValueError(f"Duplicate raw-data headers in {worksheet.Name!r}: {duplicates}")
    return headers

In [ ]:
# Run this cell after the helper-header override above. It only reads the workbooks.
uber_configuration = analyse_uber_source()

In [ ]:
def build_hr_lookups(workbook: Any) -> tuple[dict[str, str], dict[str, str], str, int]:
    """Mirror Excel XLOOKUP: the first matching HR record wins."""
    worksheet = resolve_master_sheet(workbook)
    headers = uber_headers(worksheet, 1)
    id_column = require_column(headers, "Network ID", worksheet.Name)
    name_column = require_column(headers, "Name", worksheet.Name)
    cost_center_column = require_column(headers, "Cost Center Name", worksheet.Name)
    final_row = last_populated_row(worksheet, id_column, 1)
    by_id: dict[str, str] = {}
    name_cost_centers: dict[str, set[str]] = defaultdict(set)
    for row in range_rows(worksheet, 2, final_row, max(headers.values())):
        employee_id = normalize_network_id(row[id_column - 1])
        employee_name = normalize_header(row[name_column - 1])
        cost_center = str(row[cost_center_column - 1] or "").strip()
        if employee_id and cost_center and employee_id not in by_id:
            by_id[employee_id] = cost_center
        if employee_name and cost_center:
            name_cost_centers[employee_name].add(cost_center)
    by_name = {name: next(iter(cost_centers)) for name, cost_centers in name_cost_centers.items() if len(cost_centers) == 1}
    return by_id, by_name, worksheet.Name, final_row


# Recalculate the candidate with the same first-match lookup semantics as Excel.
uber_configuration = analyse_uber_source()

In [ ]:
def next_uber_output_path() -> Path:
    OUTPUT_DIR.mkdir(exist_ok=True)
    base = OUTPUT_DIR / f"{MASTER_FILE.stem}_UBER_REFRESHED{MASTER_FILE.suffix}"
    candidate = base
    index = 1
    while candidate.exists():
        candidate = base.with_name(f"{base.stem}_{index}{base.suffix}")
        index += 1
    return candidate


# The original reload prototype remains above for reference. The final definition
# is in the next cell and is run only by the last notebook cell.

In [ ]:
def first_formula_template(worksheet: Any, column: int, first_row: int, last_row: int) -> str | None:
    for row in range(first_row, last_row + 1):
        formula = worksheet.Cells(row, column).FormulaR1C1
        if isinstance(formula, str) and formula.startswith("="):
            return formula
    return None


def original_in_scope_cost_centers(workbook: Any) -> set[str]:
    worksheet = workbook.Worksheets(IN_SCOPE_SHEET)
    marker_row = next(
        row for row in range(1, worksheet.UsedRange.Row + worksheet.UsedRange.Rows.Count)
        if normalize_header(worksheet.Cells(row, 1).Value2) == "uber lob mapping"
    )
    headers = uber_headers(worksheet, 3)
    cost_center_column = require_column(headers, "Cost Center", worksheet.Name)
    values = range_rows(worksheet, 4, marker_row - 2, max(headers.values()))
    return {normalize_header(row[cost_center_column - 1]) for row in values if row[cost_center_column - 1] not in (None, "")}


def analyse_uber_source_with_original_formulas() -> dict[str, Any]:
    excel = master_book = source_book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        master_book = excel.Workbooks.Open(str(MASTER_FILE.resolve()), ReadOnly=True)
        source_book = excel.Workbooks.Open(str(UBER_SOURCE_FILE.resolve()), ReadOnly=True)
        target = master_book.Worksheets(UBER_REPORT_SHEET)
        source = source_book.Worksheets(UBER_SOURCE_SHEET)
        target_headers = uber_headers(target, UBER_HEADER_ROW)
        source_headers = uber_headers(source, 1)
        missing = sorted(REQUIRED_UBER_SOURCE_HEADERS - set(source_headers))
        if missing:
            raise ValueError(f"Uber source is missing required columns: {missing}")
        prior_total, prior_rows, prior_last_row = current_uber_ytd(target, target_headers)
        hr_by_id, hr_by_name, hr_sheet_name, hr_last_row = build_hr_lookups(master_book)
        _, lob_header_row, lob_last_row, _ = build_lob_lookup(master_book)
        in_scope_cost_centers = original_in_scope_cost_centers(master_book)
        source_amount_column = require_column(source_headers, "Transaction Amount USD", source.Name)
        source_id_column = require_column(source_headers, "Employee ID", source.Name)
        source_first_name_column = require_column(source_headers, "First Name", source.Name)
        source_last_name_column = require_column(source_headers, "Last Name", source.Name)
        source_rows = range_rows(source, 2, last_populated_row(source, source_amount_column, 1), max(source_headers.values()))
        candidate_total = Decimal("0")
        id_matches = name_matches = unmatched = 0
        for row in source_rows:
            employee_id = normalize_network_id(row[source_id_column - 1])
            full_name = normalize_header(f"{row[source_first_name_column - 1] or ''} {row[source_last_name_column - 1] or ''}")
            cost_center = hr_by_id.get(employee_id, "") or hr_by_name.get(full_name, "")
            if hr_by_id.get(employee_id, ""):
                id_matches += 1
            elif cost_center:
                name_matches += 1
            else:
                unmatched += 1
            normalized_cost_center = normalize_header(cost_center)
            if normalized_cost_center in in_scope_cost_centers or normalize_header(cost_center.replace("US Banking Leveraged", "US Banking - Leveraged")) in in_scope_cost_centers:
                candidate_total += decimal_amount(row[source_amount_column - 1])
        unmatched_percent = Decimal(unmatched) * Decimal("100") / Decimal(len(source_rows)) if source_rows else Decimal("0")
        result = {
            "target_headers": target_headers,
            "source_headers": source_headers,
            "source_rows": source_rows,
            "prior_total": prior_total,
            "prior_rows": prior_rows,
            "prior_last_row": prior_last_row,
            "candidate_total": candidate_total,
            "id_matches": id_matches,
            "name_matches": name_matches,
            "unmatched": unmatched,
            "unmatched_percent": unmatched_percent,
            "hr_sheet_name": hr_sheet_name,
            "hr_last_row": hr_last_row,
            "lob_header_row": lob_header_row,
            "lob_last_row": lob_last_row,
        }
        print(f"Uber Report prior in-scope YTD: ${prior_total:,.2f} across {prior_rows:,} rows.")
        print(f"U4B candidate in-scope YTD: ${candidate_total:,.2f} across {len(source_rows):,} imported rows.")
        print(f"Matches: employee ID={id_matches:,}; full-name fallback={name_matches:,}; unmatched={unmatched:,} ({unmatched_percent:.2f}%).")
        if candidate_total < prior_total:
            raise RuntimeError(f"UBER REFRESH FAILED: Current in-scope YTD ${candidate_total:,.2f} is lower than prior ${prior_total:,.2f}. No output workbook was created.")
        return result
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if master_book is not None:
            master_book.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


def refresh_uber_raw_data(configuration: dict[str, Any]) -> Path:
    output_path = next_uber_output_path()
    shutil.copy2(MASTER_FILE, output_path)
    excel = master_book = source_book = None
    succeeded = False
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.ScreenUpdating = False
        excel.EnableEvents = False
        set_excel_calculation(excel, XL_CALCULATION_MANUAL)
        master_book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=False)
        source_book = excel.Workbooks.Open(str(UBER_SOURCE_FILE.resolve()), ReadOnly=True)
        target = master_book.Worksheets(UBER_REPORT_SHEET)
        source = source_book.Worksheets(UBER_SOURCE_SHEET)
        target_headers = uber_headers(target, UBER_HEADER_ROW)
        source_headers = uber_headers(source, 1)
        source_amount_column = require_column(source_headers, "Transaction Amount USD", source.Name)
        source_rows = range_rows(source, 2, last_populated_row(source, source_amount_column, 1), max(source_headers.values()))
        if len(source_rows) != len(configuration["source_rows"]):
            raise AssertionError("Uber source row count changed after pre-load validation; rerun the validation cell.")
        target_amount_column = require_column(target_headers, "Transaction Amount USD", target.Name)
        old_last_row = last_populated_row(target, target_amount_column, UBER_HEADER_ROW)
        final_row = UBER_HEADER_ROW + len(source_rows)
        final_column = max(target_headers.values())
        if final_row > old_last_row:
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, 1), target.Cells(UBER_FIRST_DATA_ROW, final_column)).Copy()
            target.Range(target.Cells(old_last_row + 1, 1), target.Cells(final_row, final_column)).PasteSpecial(Paste=XL_PASTE_FORMATS)
            target.Application.CutCopyMode = False
        for header, source_column in source_headers.items():
            if header in PROTECTED_HELPERS or header not in target_headers:
                continue
            target_column = target_headers[header]
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, target_column), target.Cells(max(old_last_row, final_row), target_column)).ClearContents()
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, target_column), target.Cells(final_row, target_column)).Value2 = tuple((row[source_column - 1],) for row in source_rows)

        full_name_column = require_column(target_headers, "Full Name", target.Name)
        employee_id_column = require_column(target_headers, "Employee ID", target.Name)
        first_name_column = require_column(target_headers, "First Name", target.Name)
        last_name_column = require_column(target_headers, "Last Name", target.Name)
        cost_center_id_column = target_headers["cost center"]
        cost_center_name_column = cost_center_id_column + 1
        protected_columns = [full_name_column, final_cost_center_column := require_column(target_headers, "Final Cost Center", target.Name), lob_column := require_column(target_headers, "LOB", target.Name), in_scope_column := require_column(target_headers, "In scope?", target.Name)]
        company_column = target_headers.get("company name")
        if company_column:
            protected_columns.append(company_column)
        template_formulas = {column: first_formula_template(target, column, UBER_FIRST_DATA_ROW, old_last_row) for column in protected_columns}
        missing_templates = [target.Cells(UBER_HEADER_ROW, column).Value2 for column, formula in template_formulas.items() if formula is None]
        if missing_templates:
            raise ValueError(f"Cannot preserve original helper formulas; no formula template found for {missing_templates}.")

        hr_name = configuration["hr_sheet_name"].replace("'", "''")
        hr_final_row = configuration["hr_last_row"]
        generated_cost_center_formulas = {
            cost_center_id_column: f"=IFERROR(XLOOKUP(TRIM(RC{employee_id_column}),'{hr_name}'!R2C13:R{hr_final_row}C13,'{hr_name}'!R2C9:R{hr_final_row}C9),\"\")",
            cost_center_name_column: f"=IFERROR(XLOOKUP(TRIM(RC{full_name_column}),'{hr_name}'!R2C12:R{hr_final_row}C12,'{hr_name}'!R2C9:R{hr_final_row}C9),\"\")",
        }
        for column, formula in generated_cost_center_formulas.items():
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(max(old_last_row, final_row), column)).ClearContents()
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(final_row, column)).FormulaR1C1 = formula
        for column, formula in template_formulas.items():
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(max(old_last_row, final_row), column)).ClearContents()
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(final_row, column)).FormulaR1C1 = formula
        set_excel_calculation(excel, XL_CALCULATION_AUTOMATIC)
        excel.CalculateFull()
        written_rows = last_populated_row(target, target_amount_column, UBER_HEADER_ROW) - UBER_HEADER_ROW
        post_total, _, _ = current_uber_ytd(target, target_headers)
        if written_rows != len(source_rows):
            raise AssertionError(f"Imported {len(source_rows):,} Uber rows but found {written_rows:,} after paste.")
        if abs(post_total - configuration["candidate_total"]) > Decimal("0.01"):
            raise AssertionError(f"Post-reload in-scope YTD ${post_total:,.2f} differs from pre-load candidate ${configuration['candidate_total']:,.2f}.")
        master_book.Save()
        succeeded = True
        print(f"Uber raw reload passed: {written_rows:,} rows; in-scope YTD ${post_total:,.2f}.")
        return output_path
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if master_book is not None:
            master_book.Close(SaveChanges=succeeded)
        if excel is not None:
            excel.Quit()
        if not succeeded and output_path.exists():
            output_path.unlink()
        gc.collect()
        pythoncom.CoUninitialize()

In [ ]:
def original_in_scope_cost_centers(workbook: Any) -> set[str]:
    """Read the CC Name list used by the original In scope? formula."""
    worksheet = workbook.Worksheets(IN_SCOPE_SHEET)
    marker_row = next(
        row for row in range(1, worksheet.UsedRange.Row + worksheet.UsedRange.Rows.Count)
        if normalize_header(worksheet.Cells(row, 1).Value2) == "uber lob mapping"
    )
    headers = uber_headers(worksheet, 3)
    cost_center_column = require_column(headers, "CC Name", worksheet.Name)
    values = range_rows(worksheet, 4, marker_row - 2, max(headers.values()))
    return {normalize_header(row[cost_center_column - 1]) for row in values if row[cost_center_column - 1] not in (None, "")}


# The pre-load calculation now exactly mirrors the original In scope? formula:
# its CC Name list, including the existing Leveraged-name substitution.
uber_configuration = analyse_uber_source_with_original_formulas()

In [ ]:
def add_cost_center_values(configuration: dict[str, Any]) -> dict[str, Any]:
    """Prepare the missing Cost Center helpers in memory before Excel is changed."""
    excel = master_book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        master_book = excel.Workbooks.Open(str(MASTER_FILE.resolve()), ReadOnly=True)
        hr_by_id, hr_by_name, _, _ = build_hr_lookups(master_book)
        source_headers = configuration["source_headers"]
        id_column = require_column(source_headers, "Employee ID", UBER_SOURCE_SHEET)
        first_name_column = require_column(source_headers, "First Name", UBER_SOURCE_SHEET)
        last_name_column = require_column(source_headers, "Last Name", UBER_SOURCE_SHEET)
        id_cost_centers = []
        name_cost_centers = []
        for row in configuration["source_rows"]:
            employee_id = normalize_network_id(row[id_column - 1])
            full_name = normalize_header(f"{row[first_name_column - 1] or ''} {row[last_name_column - 1] or ''}")
            id_cost_centers.append(hr_by_id.get(employee_id, ""))
            name_cost_centers.append(hr_by_name.get(full_name, ""))
        configuration["id_cost_centers"] = id_cost_centers
        configuration["name_cost_centers"] = name_cost_centers
        return configuration
    finally:
        if master_book is not None:
            master_book.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


def refresh_uber_raw_data(configuration: dict[str, Any]) -> Path:
    """Reload raw fields, retain original formulas, and populate missing Cost Center helpers."""
    output_path = next_uber_output_path()
    shutil.copy2(MASTER_FILE, output_path)
    excel = master_book = source_book = None
    succeeded = False
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.ScreenUpdating = False
        excel.EnableEvents = False
        set_excel_calculation(excel, XL_CALCULATION_MANUAL)
        master_book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=False)
        source_book = excel.Workbooks.Open(str(UBER_SOURCE_FILE.resolve()), ReadOnly=True)
        target = master_book.Worksheets(UBER_REPORT_SHEET)
        source = source_book.Worksheets(UBER_SOURCE_SHEET)
        target_headers = uber_headers(target, UBER_HEADER_ROW)
        source_headers = uber_headers(source, 1)
        source_amount_column = require_column(source_headers, "Transaction Amount USD", source.Name)
        source_rows = range_rows(source, 2, last_populated_row(source, source_amount_column, 1), max(source_headers.values()))
        if len(source_rows) != len(configuration["source_rows"]):
            raise AssertionError("Uber source row count changed after pre-load validation; rerun validation.")
        target_amount_column = require_column(target_headers, "Transaction Amount USD", target.Name)
        old_last_row = last_populated_row(target, target_amount_column, UBER_HEADER_ROW)
        final_row = UBER_HEADER_ROW + len(source_rows)
        final_column = max(target_headers.values())
        if final_row > old_last_row:
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, 1), target.Cells(UBER_FIRST_DATA_ROW, final_column)).Copy()
            target.Range(target.Cells(old_last_row + 1, 1), target.Cells(final_row, final_column)).PasteSpecial(Paste=XL_PASTE_FORMATS)
            target.Application.CutCopyMode = False
        for header, source_column in source_headers.items():
            if header in PROTECTED_HELPERS or header not in target_headers:
                continue
            target_column = target_headers[header]
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, target_column), target.Cells(max(old_last_row, final_row), target_column)).ClearContents()
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, target_column), target.Cells(final_row, target_column)).Value2 = tuple((row[source_column - 1],) for row in source_rows)

        cost_center_id_column = target_headers["cost center"]
        cost_center_name_column = cost_center_id_column + 1
        helper_columns = [
            require_column(target_headers, "Full Name", target.Name),
            require_column(target_headers, "Final Cost Center", target.Name),
            require_column(target_headers, "LOB", target.Name),
            require_column(target_headers, "In scope?", target.Name),
        ]
        company_column = target_headers.get("company name")
        if company_column:
            helper_columns.append(company_column)
        templates = {column: first_formula_template(target, column, UBER_FIRST_DATA_ROW, old_last_row) for column in helper_columns}
        missing = [target.Cells(UBER_HEADER_ROW, column).Value2 for column, formula in templates.items() if formula is None]
        if missing:
            raise ValueError(f"Cannot preserve original helper formulas; no template found for {missing}.")
        for column in (cost_center_id_column, cost_center_name_column):
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(max(old_last_row, final_row), column)).ClearContents()
        target.Range(target.Cells(UBER_FIRST_DATA_ROW, cost_center_id_column), target.Cells(final_row, cost_center_id_column)).Value2 = tuple((value,) for value in configuration["id_cost_centers"])
        target.Range(target.Cells(UBER_FIRST_DATA_ROW, cost_center_name_column), target.Cells(final_row, cost_center_name_column)).Value2 = tuple((value,) for value in configuration["name_cost_centers"])
        for column, formula in templates.items():
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(max(old_last_row, final_row), column)).ClearContents()
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(final_row, column)).FormulaR1C1 = formula
        set_excel_calculation(excel, XL_CALCULATION_AUTOMATIC)
        excel.CalculateFull()
        written_rows = last_populated_row(target, target_amount_column, UBER_HEADER_ROW) - UBER_HEADER_ROW
        post_total, _, _ = current_uber_ytd(target, target_headers)
        if written_rows != len(source_rows):
            raise AssertionError(f"Imported {len(source_rows):,} Uber rows but found {written_rows:,} after paste.")
        if abs(post_total - configuration["candidate_total"]) > Decimal("0.01"):
            raise AssertionError(f"Post-reload in-scope YTD ${post_total:,.2f} differs from pre-load candidate ${configuration['candidate_total']:,.2f}.")
        master_book.Save()
        succeeded = True
        print(f"Uber raw reload passed: {written_rows:,} rows; in-scope YTD ${post_total:,.2f}.")
        return output_path
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if master_book is not None:
            master_book.Close(SaveChanges=succeeded)
        if excel is not None:
            excel.Quit()
        if not succeeded and output_path.exists():
            output_path.unlink()
        gc.collect()
        pythoncom.CoUninitialize()

In [ ]:
def set_excel_calculation(excel: Any, calculation_mode: int) -> None:
    """Use a full formula rebuild when Excel policy prevents calculation-mode changes."""
    try:
        excel.Calculation = calculation_mode
    except Exception as error:
        print(f"Excel calculation mode was not changed: {error}")
    if calculation_mode == XL_CALCULATION_AUTOMATIC:
        excel.CalculateFullRebuild()

In [ ]:
def capture_uber_baseline() -> dict[str, Any]:
    """Read the input workbook YTD before any Uber data is changed."""
    excel = master_book = source_book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        master_book = excel.Workbooks.Open(str(MASTER_FILE.resolve()), ReadOnly=True)
        source_book = excel.Workbooks.Open(str(UBER_SOURCE_FILE.resolve()), ReadOnly=True)
        target = master_book.Worksheets(UBER_REPORT_SHEET)
        source = source_book.Worksheets(UBER_SOURCE_SHEET)
        target_headers = uber_headers(target, UBER_HEADER_ROW)
        source_headers = uber_headers(source, 1)
        amount_column = require_column(source_headers, "Transaction Amount USD", source.Name)
        source_rows = range_rows(source, 2, last_populated_row(source, amount_column, 1), max(source_headers.values()))
        prior_total, prior_rows, _ = current_uber_ytd(target, target_headers)
        print(f"Remembered input Uber in-scope YTD: ${prior_total:,.2f} across {prior_rows:,} rows.")
        print(f"Uber rows ready to import: {len(source_rows):,}.")
        return {"prior_total": prior_total, "target_headers": target_headers, "source_headers": source_headers, "source_rows": source_rows}
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if master_book is not None:
            master_book.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


def reload_uber_using_excel_formulas(baseline: dict[str, Any]) -> Path:
    """Reload raw Uber fields and extend the workbook's original helper formulas."""
    output_path = next_uber_output_path()
    shutil.copy2(MASTER_FILE, output_path)
    excel = master_book = source_book = None
    succeeded = False
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.ScreenUpdating = False
        excel.EnableEvents = False
        master_book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=False)
        source_book = excel.Workbooks.Open(str(UBER_SOURCE_FILE.resolve()), ReadOnly=True)
        target = master_book.Worksheets(UBER_REPORT_SHEET)
        source = source_book.Worksheets(UBER_SOURCE_SHEET)
        target_headers = uber_headers(target, UBER_HEADER_ROW)
        source_headers = uber_headers(source, 1)
        source_rows = baseline["source_rows"]
        amount_column = require_column(target_headers, "Transaction Amount USD", target.Name)
        old_last_row = last_populated_row(target, amount_column, UBER_HEADER_ROW)
        final_row = UBER_HEADER_ROW + len(source_rows)
        final_column = max(target_headers.values())
        if final_row > old_last_row:
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, 1), target.Cells(UBER_FIRST_DATA_ROW, final_column)).Copy()
            target.Range(target.Cells(old_last_row + 1, 1), target.Cells(final_row, final_column)).PasteSpecial(Paste=XL_PASTE_FORMATS)
            target.Application.CutCopyMode = False
        for header, source_column in source_headers.items():
            if header in PROTECTED_HELPERS or header not in target_headers:
                continue
            target_column = target_headers[header]
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, target_column), target.Cells(max(old_last_row, final_row), target_column)).ClearContents()
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, target_column), target.Cells(final_row, target_column)).Value2 = tuple((row[source_column - 1],) for row in source_rows)

        cost_center_id_column = target_headers["cost center"]
        cost_center_name_column = cost_center_id_column + 1
        template_columns = [
            require_column(target_headers, "Full Name", target.Name),
            require_column(target_headers, "Final Cost Center", target.Name),
            require_column(target_headers, "LOB", target.Name),
            require_column(target_headers, "In scope?", target.Name),
            target_headers.get("company name"),
        ]
        template_columns = [column for column in template_columns if column]
        templates = {column: first_formula_template(target, column, UBER_FIRST_DATA_ROW, old_last_row) for column in template_columns}
        missing = [target.Cells(UBER_HEADER_ROW, column).Value2 for column, formula in templates.items() if formula is None]
        if missing:
            raise ValueError(f"Original helper formula template missing for {missing}.")
        employee_id_column = require_column(target_headers, "Employee ID", target.Name)
        full_name_column = require_column(target_headers, "Full Name", target.Name)
        hr_sheet = resolve_master_sheet(master_book).Name.replace("'", "''")
        hr_last_row = last_populated_row(resolve_master_sheet(master_book), 13, 1)
        generated = {
            cost_center_id_column: f"=IFERROR(XLOOKUP(TRIM(RC{employee_id_column}),'{hr_sheet}'!R2C13:R{hr_last_row}C13,'{hr_sheet}'!R2C9:R{hr_last_row}C9),\"\")",
            cost_center_name_column: f"=IFERROR(XLOOKUP(TRIM(RC{full_name_column}),'{hr_sheet}'!R2C12:R{hr_last_row}C12,'{hr_sheet}'!R2C9:R{hr_last_row}C9),\"\")",
        }
        for column, formula in generated.items():
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(max(old_last_row, final_row), column)).ClearContents()
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(final_row, column)).FormulaR1C1 = formula
        for column, formula in templates.items():
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(max(old_last_row, final_row), column)).ClearContents()
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(final_row, column)).FormulaR1C1 = formula
        excel.CalculateFullRebuild()
        written_rows = last_populated_row(target, amount_column, UBER_HEADER_ROW) - UBER_HEADER_ROW
        post_total, _, _ = current_uber_ytd(target, target_headers)
        if written_rows != len(source_rows):
            raise AssertionError(f"Imported {len(source_rows):,} Uber rows but found {written_rows:,} after paste.")
        if post_total < baseline["prior_total"]:
            raise RuntimeError(f"UBER REFRESH FAILED: final in-scope YTD ${post_total:,.2f} is below input YTD ${baseline['prior_total']:,.2f}. Output was not saved.")
        master_book.Save()
        succeeded = True
        print(f"Uber reload passed: {written_rows:,} rows; final in-scope YTD ${post_total:,.2f}.")
        return output_path
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if master_book is not None:
            master_book.Close(SaveChanges=succeeded)
        if excel is not None:
            excel.Quit()
        if not succeeded and output_path.exists():
            output_path.unlink()
        gc.collect()
        pythoncom.CoUninitialize()

In [ ]:
# UBER TRANSFER TEST
# Copies source fields by header and extends existing Excel helper formulas.
# The YTD sanity check is intentionally disabled for this test.
uber_transfer = prepare_uber_transfer()
uber_output_path = transfer_uber_raw_data(uber_transfer)
print(f"Saved Uber transfer test workbook: {uber_output_path}")

In [ ]:
def prepare_uber_transfer() -> dict[str, Any]:
    """Read the monthly Uber extract without applying a YTD sanity check."""
    excel = source_book = None
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        source_book = excel.Workbooks.Open(str(UBER_SOURCE_FILE.resolve()), ReadOnly=True)
        source = source_book.Worksheets(UBER_SOURCE_SHEET)
        source_headers = uber_headers(source, 1)
        amount_column = require_column(source_headers, "Transaction Amount USD", source.Name)
        source_rows = range_rows(source, 2, last_populated_row(source, amount_column, 1), max(source_headers.values()))
        print(f"Uber rows ready to import: {len(source_rows):,}.")
        return {"source_headers": source_headers, "source_rows": source_rows}
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if excel is not None:
            excel.Quit()
        pythoncom.CoUninitialize()


def transfer_uber_raw_data(transfer: dict[str, Any]) -> Path:
    """Copy raw Uber fields by header and preserve existing helper formulas without YTD validation."""
    output_path = next_uber_output_path()
    shutil.copy2(MASTER_FILE, output_path)
    excel = master_book = source_book = None
    succeeded = False
    pythoncom.CoInitialize()
    try:
        excel = win32.DispatchEx("Excel.Application")
        excel.Visible = False
        excel.DisplayAlerts = False
        excel.ScreenUpdating = False
        excel.EnableEvents = False
        master_book = excel.Workbooks.Open(str(output_path.resolve()), ReadOnly=False)
        source_book = excel.Workbooks.Open(str(UBER_SOURCE_FILE.resolve()), ReadOnly=True)
        target = master_book.Worksheets(UBER_REPORT_SHEET)
        source = source_book.Worksheets(UBER_SOURCE_SHEET)
        target_headers = uber_headers(target, UBER_HEADER_ROW)
        source_headers = uber_headers(source, 1)
        source_rows = transfer["source_rows"]
        target_amount_column = require_column(target_headers, "Transaction Amount USD", target.Name)
        old_last_row = last_populated_row(target, target_amount_column, UBER_HEADER_ROW)
        final_row = UBER_HEADER_ROW + len(source_rows)
        final_column = max(target_headers.values())
        if final_row > old_last_row:
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, 1), target.Cells(UBER_FIRST_DATA_ROW, final_column)).Copy()
            target.Range(target.Cells(old_last_row + 1, 1), target.Cells(final_row, final_column)).PasteSpecial(Paste=XL_PASTE_FORMATS)
            target.Application.CutCopyMode = False
        for header, source_column in source_headers.items():
            if header in PROTECTED_HELPERS or header not in target_headers:
                continue
            target_column = target_headers[header]
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, target_column), target.Cells(max(old_last_row, final_row), target_column)).ClearContents()
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, target_column), target.Cells(final_row, target_column)).Value2 = tuple((row[source_column - 1],) for row in source_rows)

        helper_columns = [
            require_column(target_headers, "Full Name", target.Name),
            require_column(target_headers, "Final Cost Center", target.Name),
            require_column(target_headers, "LOB", target.Name),
            require_column(target_headers, "In scope?", target.Name),
        ]
        company_column = target_headers.get("company name")
        if company_column:
            helper_columns.append(company_column)
        templates = {column: first_formula_template(target, column, UBER_FIRST_DATA_ROW, old_last_row) for column in helper_columns}
        missing_templates = [target.Cells(UBER_HEADER_ROW, column).Value2 for column, formula in templates.items() if formula is None]
        if missing_templates:
            raise ValueError(f"Original helper formula template missing for {missing_templates}.")
        for column, formula in templates.items():
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(max(old_last_row, final_row), column)).ClearContents()
            target.Range(target.Cells(UBER_FIRST_DATA_ROW, column), target.Cells(final_row, column)).FormulaR1C1 = formula

        excel.CalculateFullRebuild()
        written_rows = last_populated_row(target, target_amount_column, UBER_HEADER_ROW) - UBER_HEADER_ROW
        if written_rows != len(source_rows):
            raise AssertionError(f"Imported {len(source_rows):,} Uber rows but found {written_rows:,} after paste.")
        master_book.Save()
        succeeded = True
        print(f"Uber transfer passed: {written_rows:,} source rows copied. No YTD sanity check was run.")
        return output_path
    finally:
        if source_book is not None:
            source_book.Close(SaveChanges=False)
        if master_book is not None:
            master_book.Close(SaveChanges=succeeded)
        if excel is not None:
            excel.Quit()
        if not succeeded and output_path.exists():
            output_path.unlink()
        gc.collect()
        pythoncom.CoUninitialize()